# Phase 3: 模型验证 (Model Validation)

## 概述 Overview

本Notebook实现Phase 3的二分类建模与迁移学习验证。
这段代码是否需要单独进行处理

**主要任务：**
1. **标签二值化**: 基于中位数将稳定性分为"稳定"vs"不稳定"
2. **迁移学习**: 评估模型在不同数据集间的泛化能力


**模型更新内容如下**
- npz中的ids不能再与csv中的id实现对应了，需要默认按照内容进行一一对齐
- 不再考虑筛选内容（Phase3中仅考虑异常值）
- 由于数据集已经拆分，每次只能做一种表征和一种任务了


**超参数配置：**
- Logistic Regression: L2正则化, balanced权重
- Random Forest: 100棵树, balanced权重
- XGBoost: GPU加速, max_depth=6, balanced权重

**数据配置：**
- 半衰期阈值：SIF：270，SGF：250
- 半衰期异常值：700
- 筛选条件：（在前面已经进行过筛选 ）
- 随机规则：**随机种子**+**每次随机洗牌数据集顺序**

**输入**: 
- `data/{实际需求情况}/csv/*.csv` - 带分钟标签的CSV  
- `data/{实际需求情况}/features/*.npz` - RDKit特征矩阵  

**输出**: 
- `data/{实际需求情况}/output/*.npz`

---

## 1. 环境检查与导入 Environment Setup

In [9]:
# 环境检查
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
print(project_root)
sys.path.insert(0, str(project_root / "src"))

# 核心库导入
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json
warnings.filterwarnings('ignore')

# 机器学习
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

# 检查GPU可用性
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"✓ GPU可用: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ GPU不可用，将使用CPU训练")
except ImportError:
    gpu_available = False
    print("⚠ PyTorch未安装，将使用CPU训练")

# 设置显示选项
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ 所有库已成功导入")
print(f"✓ 项目根目录: {project_root}")

d:\RA\feature_extraction
⚠ GPU不可用，将使用CPU训练
✓ 所有库已成功导入
✓ 项目根目录: d:\RA\feature_extraction


In [10]:

def convert_numpy_types(obj):
    """递归转换numpy类型为Python原生类型"""
    import numpy as np
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(v) for v in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj


## 2. 参数配置区 Configuration

**⚙️ 根据您的需求修改以下参数**

- 对于迁移操作来说，下面的配置都是不能改的，因为我们先前的工作全都是这个

In [11]:
# ============== 参数配置区 ==============

CONFIG = {
    # 输入输出路径
    'processed_dir': project_root / 'data' / 'processed',
    'features_dir': project_root / 'outputs' / 'features',
    'cv_results_dir': project_root / 'outputs' / 'model_results' / 'phase3_binary' / 'cv_results',
    'feature_importance_dir': project_root / 'outputs' / 'model_results' / 'phase3_binary' / 'feature_importance',
    'transfer_results_dir': project_root / 'outputs' / 'model_results' / 'phase3_binary' / 'transfer_results',
    'figures_dir': project_root / 'outputs' / 'figures' / 'phase3',

    'message':"Morgan_Avalon",
    
    # 模型选择（可选: 'lr', 'rf', 'xgb'）
    'models_to_train': ['lr', 'rf', 'xgb'],
    
    # 交叉验证参数
    'n_folds': 5,
    'random_state': 42,
    
    # XGBoost参数
    'use_gpu': gpu_available,
    'xgb_max_depth': 6,
    'xgb_learning_rate': 0.1,
    'xgb_n_estimators': 100,
    
    # Random Forest参数
    'rf_n_estimators': 100,
    'rf_n_jobs': -1,
    
    # Logistic Regression参数
    'lr_max_iter': 1000,
    
    # 可视化参数
    'dpi': 300,
    'format': 'png',
    'display_plots': True,
    'max_display_plots': 8,
}

# 创建输出目录
for key in ['cv_results_dir', 'feature_importance_dir', 'transfer_results_dir', 'figures_dir']:
    CONFIG[key].mkdir(parents=True, exist_ok=True)

print("配置参数:")
print(f"  模型: {CONFIG['models_to_train']}")
print(f"  交叉验证折数: {CONFIG['n_folds']}")
print(f"  GPU加速: {CONFIG['use_gpu']}")
print(f"  XGBoost参数: max_depth={CONFIG['xgb_max_depth']}, lr={CONFIG['xgb_learning_rate']}")

配置参数:
  模型: ['lr', 'rf', 'xgb']
  交叉验证折数: 5
  GPU加速: False
  XGBoost参数: max_depth=6, lr=0.1


## 3. 数据加载与二值化 Data Loading & Binarization

In [12]:
def load_and_binarize_dataset(npz_path: Path, csv_path: Path, target: str, is_monomer: bool = None):
    """
    加载数据并将标签二值化，采用索引顺序匹配而非ID匹配
    (2026-01-16 更新: 解决ID重复问题，直接按行顺序对应)
    
    Args:
        npz_path: NPZ特征文件 (包含 X, y_sif, y_sgf 等)
        csv_path: 处理后的CSV文件 (包含标签和筛选列)
        target: 'SIF' or 'SGF'
        is_monomer: 如果为True，只保留monomer；False，只保留非monomer；None不筛选
    
    Returns:
        X_valid, y_binary, median, feature_names
    """
    # 1. 加载NPZ特征
    data = np.load(npz_path, allow_pickle=True)
    X = data['X']
    feature_names = data['feature_names']
    # 注意：这里不再提取 ids_npz，因为我们直接按物理顺序读取
    
    # 2. 加载CSV获取标签及筛选条件
    df = pd.read_csv(csv_path)
    
    # 3. 顺序匹配逻辑
    # 既然 npz 的 X 属性与 csv 默认顺序对应，我们直接根据 df 的索引进行筛选
    valid_indices = []
    valid_labels = []
    
    label_col = f"{target}_minutes"
    
    # 使用 zip 或直接遍历索引，确保 X 的第 i 行对应 df 的第 i 行
    for i, row in df.iterrows():
        # 获取当前行的标签
        label = row[label_col]
        
        # =================================== 过滤操作, 虽然已经操作过了 ===================================
        # 1. 标签有效性判断
        if label == -1 or pd.isna(label):
            continue
            
        # 2. monomer 条件筛选
        if is_monomer is not None and row['is_monomer'] != is_monomer:
            continue
            
        # 3. 特殊阈值处理 (SIF/SGF_minutes > 700 排除)
        if label > 700:
            continue
        # ===============================================================================
        
        # 记录通过筛选的行索引和对应的标签
        valid_indices.append(i) # 选择合理的X
        valid_labels.append(label) # 选择合理的y
    
    # 4. 筛选有效样本
    # 利用 numpy 的高级索引，根据保存的行索引一次性提取对应的特征行
    X_valid = X[valid_indices]
    y_minutes = np.array(valid_labels)

    
    
    # 设定阈值========================
    median =None
    if target=='SIF':
        median = 270
    elif target=='SGF':
        median =250   #（经过纠正250更好点）
    
    # 根据阈值判断是否稳定捏
    y_binary = (y_minutes >= median).astype(int)  # 1=稳定, 0=不稳定
    
    print(f"  样本数: {len(X_valid)}")
    print(f"  中位数阈值（更新后阈值根据75%划分得出结果）: {median:.1f} 分钟")
    print(f"  稳定/不稳定: {np.sum(y_binary==1)}/{np.sum(y_binary==0)}")
    
    return X_valid, y_binary, median, feature_names


# 加载所有数据集
datasets_data = {}
npz_files = sorted(CONFIG['features_dir'].glob('*_processed.npz'))
# 选择加载数据的时候就指定monomer或非monomer样本
is_monomer = True # 仅加载monomer样本，设置为False则加载非monomer样本，None则不筛选

print(f"加载并二值化 {len(npz_files)} 个数据集:\n")
for npz_file in npz_files:
    dataset_name = npz_file.stem.replace('_processed', '')
    csv_file = CONFIG['processed_dir'] / f"{dataset_name}_processed.csv"
    
    print(f"{dataset_name}:")
    X_sif, y_sif, median_sif, feat_names = load_and_binarize_dataset(npz_file, csv_file, 'SIF',is_monomer)
    print(f"  SIF数据选择完成，只获得单体数据")
    X_sgf, y_sgf, median_sgf, _ = load_and_binarize_dataset(npz_file, csv_file, 'SGF',is_monomer)
    print(f"  SGF数据选择完成，只获取单体数据\n")
    
    datasets_data[dataset_name] = {
        'X_sif': X_sif,
        'y_sif': y_sif,
        'median_sif': median_sif,
        'X_sgf': X_sgf,
        'y_sgf': y_sgf,
        'median_sgf': median_sgf,
        'feature_names': feat_names,
    }

print(f"✓ 数据加载完成！共 {len(datasets_data)} 个数据集")



加载并二值化 5 个数据集:

sif_sgf_second:
  样本数: 202
  中位数阈值（更新后阈值根据75%划分得出结果）: 270.0 分钟
  稳定/不稳定: 0/202
  SIF数据选择完成，只获得单体数据
  样本数: 202
  中位数阈值（更新后阈值根据75%划分得出结果）: 250.0 分钟
  稳定/不稳定: 0/202
  SGF数据选择完成，只获取单体数据

US20140294902A1:
  样本数: 5
  中位数阈值（更新后阈值根据75%划分得出结果）: 270.0 分钟
  稳定/不稳定: 3/2
  SIF数据选择完成，只获得单体数据
  样本数: 0
  中位数阈值（更新后阈值根据75%划分得出结果）: 250.0 分钟
  稳定/不稳定: 0/0
  SGF数据选择完成，只获取单体数据

US9624268:
  样本数: 130
  中位数阈值（更新后阈值根据75%划分得出结果）: 270.0 分钟
  稳定/不稳定: 76/54
  SIF数据选择完成，只获得单体数据
  样本数: 90
  中位数阈值（更新后阈值根据75%划分得出结果）: 250.0 分钟
  稳定/不稳定: 52/38
  SGF数据选择完成，只获取单体数据

US9809623B2:
  样本数: 26
  中位数阈值（更新后阈值根据75%划分得出结果）: 270.0 分钟
  稳定/不稳定: 5/21
  SIF数据选择完成，只获得单体数据
  样本数: 5
  中位数阈值（更新后阈值根据75%划分得出结果）: 250.0 分钟
  稳定/不稳定: 5/0
  SGF数据选择完成，只获取单体数据

WO2017011820A2:
  样本数: 150
  中位数阈值（更新后阈值根据75%划分得出结果）: 270.0 分钟
  稳定/不稳定: 92/58
  SIF数据选择完成，只获得单体数据
  样本数: 105
  中位数阈值（更新后阈值根据75%划分得出结果）: 250.0 分钟
  稳定/不稳定: 65/40
  SGF数据选择完成，只获取单体数据

✓ 数据加载完成！共 5 个数据集


## 4. 函数选取

新增：模型选取的时候自动使用随机种子

新增：自动调整数据集顺序

In [13]:
import random
def get_model(model_name: str, use_gpu: bool = False):
    """
    创建模型实例
    """
    if model_name == 'lr':
        return LogisticRegression(
            max_iter=CONFIG['lr_max_iter'],
            class_weight='balanced',
            random_state=random.randint(1, 1000000)#  CONFIG['random_state']
        )
    elif model_name == 'rf':
        return RandomForestClassifier(
            n_estimators=CONFIG['rf_n_estimators'],
            class_weight='balanced',
            n_jobs=CONFIG['rf_n_jobs'],
            random_state=random.randint(1, 1000000)#CONFIG['random_state']
        )
    elif model_name == 'xgb':
        params = {
            'max_depth': CONFIG['xgb_max_depth'],
            'learning_rate': CONFIG['xgb_learning_rate'],
            'n_estimators': CONFIG['xgb_n_estimators'],
            'random_state': random.randint(1, 1000000),#CONFIG['random_state'],
            'tree_method': 'hist',
        }
        if use_gpu:
            params['device'] = 'cuda:0'
        return XGBClassifier(**params)
    else:
        raise ValueError(f"Unknown model: {model_name}")


## 5. 迁移学习 Transfer Learning

评估模型在不同数据集间的泛化能力（Train on A, Test on B）

生成json的结构如下：


In [ ]:

const json_format={
  "train_dataset": "string",        // source dataset name
  "test_dataset": "string",         // target dataset name
  "target": "SIF" | "SGF",           // prediction task
  "model": "string",                 // model name
  "runs": [                           // per-run results (length ≤ n_runs)
    {
      "accuracy": float,
      "precision": float,
      "recall": float,
      "f1": float,
      "confusion_matrix": [[int, int], [int, int]]
    },
    ...
  ],
  "mean": {                           // mean over all successful runs
    "accuracy": float | NaN,
    "precision": float | NaN,
    "recall": float | NaN,
    "f1": float | NaN,
    "confusion_matrix": [[float, float], [float, float]] | null
  }
}

In [14]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json  # 新增

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# =====================================================
# 1. 单次迁移训练函数
# =====================================================

def transfer_learning_test(
    X_train, y_train,
    X_test, y_test,
    model_name: str,
    use_gpu: bool = False
):
    """
    单次 source → target 迁移学习
    """
    model = get_model(model_name, use_gpu)

    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    except Exception:
        return None

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        #"auc":roc_auc_score(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
    }

# ===========================
# 1.1 多次进行迁移学习并且输出均值
# ===========================

def transfer_learning_test_mean(
    X_train, y_train,
    X_test, y_test,
    model_name: str,
    use_gpu: bool = False,
    n_runs: int = 10,
    random_seed: int = 42
):
    """
    多次 source → target 迁移学习
    返回：
        {
            "runs": [ {单次指标}, ... ],
            "mean": {均值指标}
        }
    若全部失败，返回 None
    """
    run_metrics = []

    for i in range(n_runs):
        model = get_model(model_name, use_gpu)

        if hasattr(model, "random_state"):
            model.random_state = random_seed + i

        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        except Exception as e:
            print(f"  ⚠ 第 {i} 次迁移失败：{e}")
            continue

        run_metrics.append({
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "confusion_matrix": confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()
        })

    if len(run_metrics) == 0:
        return None

    mean_metrics = {
        k: float(np.nanmean([m[k] for m in run_metrics]))
        for k in ["accuracy", "precision", "recall", "f1"]
    }
    mean_metrics["confusion_matrix"] = (
        np.mean([m["confusion_matrix"] for m in run_metrics], axis=0)
        .tolist()
    )

    return {
        "runs": run_metrics,
        "mean": mean_metrics
    }
print("\n开始全数据集迁移学习实验...\n")

transfer_results = []

dataset_names = list(datasets_data.keys())

for train_dataset in dataset_names:
    for test_dataset in dataset_names:
        if train_dataset == test_dataset:
            continue

        print(f"\n▶ {train_dataset} → {test_dataset}")

        for target in ["SIF", "SGF"]:
            X_train = datasets_data[train_dataset][f"X_{target.lower()}"]
            y_train = datasets_data[train_dataset][f"y_{target.lower()}"]

            X_test = datasets_data[test_dataset][f"X_{target.lower()}"]
            y_test = datasets_data[test_dataset][f"y_{target.lower()}"]

            if len(y_train) == 0 or len(y_test) == 0:
                print(f"  ⚠ 跳过 {target}（空数据）")
                continue

            for model_name in CONFIG["models_to_train"]:
                print(f"  - {target} | {model_name}")

                metrics = transfer_learning_test_mean(
                    X_train, y_train,
                    X_test, y_test,
                    model_name,
                    use_gpu=CONFIG["use_gpu"],
                    n_runs=CONFIG.get("n_runs", 10)
                )

                # 若失败，明确记为 nan
                if metrics is None:
                    result = {
                        "train_dataset": train_dataset,
                        "test_dataset": test_dataset,
                        "target": target,
                        "model": model_name,
                        "runs": [],
                        "mean": {
                            "accuracy": np.nan,
                            "precision": np.nan,
                            "recall": np.nan,
                            "f1": np.nan,
                            "confusion_matrix": None
                        }
                    }
                else:
                    result = {
                        "train_dataset": train_dataset,
                        "test_dataset": test_dataset,
                        "target": target,
                        "model": model_name,
                        "runs": metrics["runs"],
                        "mean": metrics["mean"]
                    }

                transfer_results.append(result)

print(f"\n✓ 迁移实验完成，共 {len(transfer_results)} 个迁移任务\n")
output_path = f"transfer_{CONFIG['message']}.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(transfer_results, f, ensure_ascii=False, indent=2)

print(f"✓ 迁移结果已保存到 {output_path}\n")



开始全数据集迁移学习实验...


▶ sif_sgf_second → US20140294902A1
  - SIF | lr
  ⚠ 第 0 次迁移失败：This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)
  ⚠ 第 1 次迁移失败：This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)
  ⚠ 第 2 次迁移失败：This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)
  ⚠ 第 3 次迁移失败：This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)
  ⚠ 第 4 次迁移失败：This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)
  ⚠ 第 5 次迁移失败：This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)
  ⚠ 第 6 次迁移失败：This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)
  ⚠ 第 7 次迁移失败：This solver needs samples of at least 2 clas